# Fusion products and extensions

This notebook introduces two constructions on modules over a vertex algebra: the fusion product $M \boxtimes N$, computed by the Nahm–Gaberdiel–Kausch algorithm, and the extensions between modules, computed from projective covers. Every example is small and has a known answer: the Lee–Yang and Ising minimal models, affine $\mathfrak{sl}_2$ at level 1, the first logarithmic fusion product at $c = -2$, and a Jordan block of $L_0$ on a Virasoro module. The research examples come afterwards, in `15_triplet_modules.ipynb` (the triplet algebra $\mathcal W(2)$) and `16_virasoro_c0_fusion.ipynb` (Virasoro fusion at $c = 0$).

**Prerequisites:** `01_getting_started.ipynb`, `09_zhu_algebras.ipynb`, `10_modules.ipynb`.

**Contents**

1. Fusion by the Nahm–Gaberdiel–Kausch algorithm
2. The window
3. More rational examples: Ising and affine $\mathfrak{sl}_2$ at level 1
4. A logarithmic fusion product at $c = -2$
5. Projective covers and $\operatorname{Ext}^1$: the semisimple case
6. A non-split extension: a Jordan block of $L_0$
7. Not there yet

## 1. Fusion by the Nahm–Gaberdiel–Kausch algorithm

The fusion product $M \boxtimes N$ of two modules over a vertex algebra $V$ is a quotient of $M \otimes N$ on which $V$ acts through a coproduct. The library uses the conventions of Kanade and Ridout [KR2019]: $M$ sits at $w = 1$ and $N$ at $w = 0$, and a mode acts by

$$\Delta\bigl(v_{(k)}\bigr) = \sum_{j \ge 0} \binom{k}{j}\, v_{(j)} \otimes 1 + 1 \otimes v_{(k)},$$

so for the Virasoro field $\Delta(L_0) = L_{-1} \otimes 1 + L_0 \otimes 1 + 1 \otimes L_0$. The whole product is infinite, so the library computes its depth-$d$ quotient $(M \boxtimes N)^{(d)}$ [Nah1994], [GK1996]: $M \boxtimes N$ modulo the span of all words of modes of total degree $> d$ (the degree of a mode is how much it raises the conformal weight). This is a finite-dimensional space on which $L_0$ and the modes of degree $0, \dots, d$ act. For a direct sum of highest weight modules it is the sum of their weight spaces of depth $\le d$, each counted from its own lowest weight.

The Lee–Yang model is the Virasoro minimal model at $c = -\frac{22}{5}$, with two simple modules, of lowest weights $0$ and $-\frac15$ [BPZ1984]. Its fusion rule is $\varphi \boxtimes \varphi = 1 + \varphi$, for $\varphi = L(-\frac15)$. `M.fusion_product(N)` builds the product (nothing is computed yet); `depth_quotient_up_to(d, max_depth=K)` computes the depth-$d$ quotient from the tensors of total depth $\le K$ (section 2).

In [1]:
import vertex_algebras as va
from vertex_algebras import *

V = va.Virasoro(QQ, -22/5, conformal_vector='L')
L = V.gen('L')
phi = V.verma_module(-1/5).simple_quotient()
one = V.verma_module(0).simple_quotient()
P = phi.fusion_product(phi)
F = P.depth_quotient_up_to(0, max_depth=3)
F.dimension(), F.basis()

(2, [(v, v), (L_{-1} v, v)])

The basis is given by representatives $x \otimes y$: here $v \otimes v$ and $L_{-1}v \otimes v$, for $v$ the lowest weight vector of $\varphi$. `l0_matrix()` is the matrix of $\Delta(L_0)$ (columns are the images of the basis vectors). It is not diagonal, because $\Delta(L_0)$ contains $L_{-1} \otimes 1$. `conformal_weights()` lists its eigenvalues $h$ with the sizes of the Jordan blocks: one state of weight $-\frac15$ and one of weight $0$, the lowest weight vectors of $\varphi$ and of $1$.

In [2]:
F.l0_matrix(), F.conformal_weights()

(
[ -2/5 -2/25]                         
[    1   1/5], [(-1/5, [1]), (0, [1])]
)

At depth 2 the quotient contains the weight spaces of depth $\le 2$ of both summands. $\varphi$ has one state at each of the weights $-\frac15, \frac45, \frac95$ and the vacuum module has states at $0$ and $2$ only ($L_{-1}\Omega = 0$):

In [3]:
P.depth_quotient_up_to(2, max_depth=6).conformal_weights()

[(-1/5, [1]), (0, [1]), (4/5, [1]), (9/5, [1]), (2, [1])]

In [4]:
[phi.weight_space(-1/5 + d).dimension() for d in range(3)], [one.weight_space(d).dimension() for d in range(3)]

([1, 1, 1], [1, 0, 1])

The vacuum module is the unit for fusion, on either side:

In [5]:
one.fusion_product(phi).depth_quotient_up_to(0, 3).conformal_weights(), phi.fusion_product(one).depth_quotient_up_to(0, 3).conformal_weights()

([(-1/5, [1])], [(-1/5, [1])])

The depth-0 quotient is a module over the Zhu algebra $A(V)$ [Zhu1996], through the zero modes. `zhu_algebra_module(A)` returns it as such, so it can be fed back into `induced_module` (see `10_modules.ipynb`). Other modes are available as matrices: `mode_matrix(a, n)` for $\Delta(a_{(n)})$, `zero_mode_matrix(a)`, and `projection(d')` to a smaller depth.

In [6]:
F.zhu_algebra_module(V.zhu_algebra())

Module of dimension 2 over Zhu algebra of Universal vertex algebra on generators ['L'] over Rational Field

## 2. The window

The computation uses only the tensors $x \otimes y$ of total depth $\le K$ = `max_depth` (depth = weight minus the lowest weight of the factor), and the relations whose terms all lie there. Nahm [Nah1994] and Gaberdiel–Kausch [GK1996] showed that the depth-$d$ quotient is spanned by $M^{ss} \otimes N^{(d)}$, where $N^{(d)}$ is the depth-$d$ quotient of $N$ and $M^{ss} = M / \sum_{b < 0} v_{(b)} M$ (sum over the strong generators $v$) is Nahm's special subspace. For the Virasoro algebra, $M^{ss} = M / \sum_{n \ge 2} L_{-n} M$. `special_subspace_dimensions()` gives its dimensions by depth, as far as the window sees them. For $\varphi$ it is spanned by $v$ and $L_{-1}v$ (by the null vector at depth 2, $L_{-1}^2 v$ is a multiple of $L_{-2} v$), which explains the bound $\dim \le 2 \cdot 1$ above.

In [7]:
F.special_subspace_dimensions()

{0: 1, 1: 1, 2: 0, 3: 0}

The answer is only reliable once the window is large enough, and the code cannot tell when that is: the result is computed "up to" `max_depth`, and checking that it is stable as `max_depth` grows is the user's job. The dimension can be too small (the special subspace is not yet in the window) or too large (the relations that cut the product down come from the null vectors of the factors, which must then lie in the window). Lee–Yang $\varphi \boxtimes \varphi$ at depth 0, for $K = 0, \dots, 5$, is too small only at $K = 0$:

In [8]:
[P.depth_quotient_up_to(0, K).dimension() for K in range(6)]

[1, 2, 2, 2, 2, 2]

In the Ising model ($c = \frac12$), $\varepsilon \boxtimes \varepsilon = 1$ for $\varepsilon = L(\frac12)$, so the depth-0 quotient has dimension 1. In small windows the result is too large: it is right from $K = 3$ on (here up to $K = 5$). The null vectors of $\varepsilon$ lie at depths 2 and 3: the maximal submodule of the Verma module $M(\frac12)$ has dimensions $0, 0, 1, 2$ at depths $0, \dots, 3$, a singular vector at depth 2 and a new generator at depth 3.

In [9]:
Vi = va.Virasoro(QQ, 1/2, conformal_vector='L')
eps = Vi.verma_module(1/2).simple_quotient()
[eps.fusion_product(eps).depth_quotient_up_to(0, K).dimension() for K in range(6)]

[1, 2, 2, 1, 1, 1]

In [10]:
[Vi.verma_module(1/2).maximal_submodule().weight_space(1/2 + d).dimension() for d in range(4)]

[0, 0, 1, 2]

## 3. More rational examples: Ising and affine $\mathfrak{sl}_2$ at level 1

The three simple modules of the Ising model have lowest weights $0, \frac12, \frac1{16}$ ($1, \varepsilon, \sigma$) [BPZ1984]. Their fusion rules are $\varepsilon \boxtimes \varepsilon = 1$, $\varepsilon \boxtimes \sigma = \sigma$ and $\sigma \boxtimes \sigma = 1 + \varepsilon$. The table of the conformal weights of the depth-0 quotients (rows and columns $1, \varepsilon, \sigma$) shows all of them:

In [11]:
ising = [Vi.verma_module(h).simple_quotient() for h in (0, 1/2, 1/16)]
def weights(A, B):
    return [h for h, sizes in A.fusion_product(B).depth_quotient_up_to(0, max_depth=3).conformal_weights()]
table([[weights(A, B) for B in ising] for A in ising], header_row=['1', 'ε', 'σ'], header_column=['', '1', 'ε', 'σ'])

,1,ε,σ
1,\(\left[0\right]\),\(\left[\frac{1}{2}\right]\),\(\left[\frac{1}{16}\right]\)
ε,\(\left[\frac{1}{2}\right]\),\(\left[0\right]\),\(\left[\frac{1}{16}\right]\)
σ,\(\left[\frac{1}{16}\right]\),\(\left[\frac{1}{16}\right]\),"\(\left[0, \frac{1}{2}\right]\)"


At depth 1, $\sigma \boxtimes \sigma$ adds the state $L_{-1}\varepsilon$ of weight $\frac32$; the vacuum module has no state of depth 1:

In [12]:
sigma = ising[2]
sigma.fusion_product(sigma).depth_quotient_up_to(1, max_depth=4).conformal_weights()

[(0, [1]), (1/2, [1]), (3/2, [1])]

Affine $\mathfrak{sl}_2$ at level 1 has two simple modules: the vacuum module $L(0)$ and $L(\omega_1)$, whose top level is the doublet, of conformal weight $\frac14$. They are the two modules $V_{\mathbb Z\alpha}$ and $V_{\alpha/2 + \mathbb Z\alpha}$ of the lattice vertex algebra of $A_1$ (`10_modules.ipynb`, sections 2 and 5), and the lattice fusion rule $V_{\lambda+L} \boxtimes V_{\mu+L} = V_{\lambda+\mu+L}$ predicts $L(\omega_1) \boxtimes L(\omega_1) = L(0)$. The fusion code needs a finite set of strong generators, which lattice vertex algebras do not provide, so we use the affine description. The modules are the simple quotients of the induced modules of the trivial module and the doublet of the Zhu algebra $U(\mathfrak{sl}_2)$:

In [13]:
A1 = va.Affine(QQ, ['A', 1], 1, names=['E', 'H', 'F'], conformal_vector='sugawara')
Z = A1.zhu_algebra()
doublet = Z.module({'E': matrix([[0, 1], [0, 0]]), 'H': matrix([[1, 0], [0, -1]]), 'F': matrix([[0, 0], [1, 0]])})
trivial = Z.module({'E': matrix(QQ, 1), 'H': matrix(QQ, 1), 'F': matrix(QQ, 1)})
D = A1.induced_module(doublet).simple_quotient()
O = A1.induced_module(trivial).simple_quotient()
D.character(4)

2*q^(1/4) + 2*q^(5/4) + 6*q^(9/4) + 8*q^(13/4) + O(q^4)

The depth-0 quotient of $L(\omega_1) \boxtimes L(\omega_1)$ is one state of weight 0 on which $H_0$ acts by 0: the top of $L(0)$. At depth 1 it adds three states of weight 1. `horizontal_grades()` gives the $\mathfrak{sl}_2$-weights of the basis states (in fundamental weight coordinates): $2, 0, 0, -2$, that is $0$ for the state of weight 0 and $2, 0, -2$ for the adjoint representation, the weight-1 space $E_{-1}\Omega, H_{-1}\Omega, F_{-1}\Omega$ of $L(0)$.

In [14]:
DD = D.fusion_product(D)
F0 = DD.depth_quotient_up_to(0, max_depth=3)
F1 = DD.depth_quotient_up_to(1, max_depth=4)
F0.conformal_weights(), F0.zero_mode_matrix(A1.gen('H')), F1.conformal_weights(), F1.horizontal_grades()

([(0, [1])], [0], [(0, [1]), (1, [1, 1, 1])], [(2,), (0,), (0,), (-2,)])

The vacuum module is again the unit: $L(0) \boxtimes L(\omega_1)$ has the doublet top of $L(\omega_1)$ at weight $\frac14$.

In [15]:
O.fusion_product(D).depth_quotient_up_to(0, max_depth=3).conformal_weights()

[(1/4, [1, 1])]

## 4. A logarithmic fusion product at $c = -2$

The same algorithm finds fusion products on which $L_0$ is not diagonalizable. Gaberdiel and Kausch [GK1996] (see also [KR2019], section 8) fused the Virasoro module $L(-\frac18)$ at $c = -2$ with itself: the depth-0 quotient is two-dimensional, with a single Jordan block of $L_0$ at weight 0.

In [16]:
Vc = va.Virasoro(QQ, -2, conformal_vector='L')
mu = Vc.verma_module(-1/8).simple_quotient()
Fc = mu.fusion_product(mu).depth_quotient_up_to(0, max_depth=3)
Fc.basis(), Fc.l0_matrix(), Fc.conformal_weights()

(
                         [ -1/4 -1/16]            
[(v, v), (L_{-1} v, v)], [    1   1/4], [(0, [2])]
)

`[(0, [2])]` reads: the generalized eigenvalue 0 with one Jordan block of size 2. Here $L(-\frac18)$ is a module over the universal Virasoro algebra. The triplet algebra $\mathcal W(2)$ at $c = -2$ has a simple module $\mathcal W(-\frac18)$ of the same lowest weight, and $\mathcal W(-\frac18) \boxtimes \mathcal W(-\frac18)$ is the projective cover $\mathcal P(0)$ of its vacuum module [GK1996]. That example, with the whole fusion table of $\mathcal W(2)$, is in `15_triplet_modules.ipynb`; the Virasoro fusion products at $c = 0$, with Jordan blocks of rank 2 and 3 and their logarithmic couplings, are in `16_virasoro_c0_fusion.ipynb`.

## 5. Projective covers and $\operatorname{Ext}^1$: the semisimple case

The modules of this section are positive energy modules over a quotient $Q$ of a universal vertex algebra, with generalized $L_0$ eigenvalues in $h + \mathbb Z_{\ge 0}$. For such a module $N$, the degree-$n$ space $N(n)$ (weight $h + n$) is a module over the level-$n$ Zhu algebra $A_n(Q)$, and $\operatorname{Hom}_V(\bar M_n(U), N) = \operatorname{Hom}_{A_n}(U, N(n))$ for the universal module $\bar M_n(U)$ induced from an $A_n$-module $U$ in degree $n$ [DLM1998], [He2017], [BVY2019]. Hence a projective $A_n$-module $U$ induces a projective module. `L.projective_cover(max_weight, max_depth)` builds $\pi: \bar M_n(A_n e) \to L$ for a simple module $L$, with $e$ a primitive idempotent of $A_n$ and the level $n$ chosen so that every module of finite length whose composition factors are simple modules of $Q$ lies in the category.

Everything is computed "up to": `max_weight` bounds the weight of the relations of $Q$ that are used (the Zhu algebras computed from them map onto the true ones), `max_depth` the depth to which modules are computed. The simple modules of the Lee–Yang vertex algebra $Q = V / \langle {:}LL{:} - \frac3{10}\partial^2 L \rangle$ (`Q.simple_modules_up_to(max_weight, max_depth)`, see `10_modules.ipynb`) are again $L(-\frac15)$ and $L(0)$ (`top_weight()` is the lowest weight; then the dimensions of the weight spaces of depth $0, \dots, 5$):

In [17]:
Q = V / [L.no(L) - 3/10*L.derivative(2)]
simples = Q.simple_modules_up_to(6, 5)
[S.top_weight() for S in simples], [[S.weight_space(S.top_weight() + d).dimension() for d in range(6)] for S in simples]

([-1/5, 0], [[1, 1, 1, 1, 2, 2], [1, 0, 1, 1, 1, 1]])

The Lee–Yang vertex algebra is rational [Wan1993], so every module is projective and each simple module is its own projective cover. The library finds that: the domains of the projective covers have the same weight spaces as the simple modules (to depth 5), and $\pi$ is onto.

In [18]:
covers = [S.projective_cover(6, 5) for S in simples]
[[pi.domain().weight_space(S.top_weight() + d).dimension() for d in range(6)] for pi, S in zip(covers, simples)], [pi.is_surjective_up_to(5) for pi in covers]

([[1, 1, 1, 1, 2, 2], [1, 0, 1, 1, 1, 1]], [True, True])

For $\pi: P \to M$ onto, $P$ projective and $K = \ker \pi$, the exact sequence $0 \to \operatorname{Hom}(M, N) \to \operatorname{Hom}(P, N) \to \operatorname{Hom}(K, N) \to \operatorname{Ext}^1(M, N) \to 0$ [Wei1994] gives $\operatorname{Ext}^1(M, N) = \operatorname{Hom}(K, N) / \{f|_K\}$. `M.ext1_up_to(N, depth, cover)` computes it with the generators of $K$ up to `depth` and maps that commute with the modes in degrees $\le$ `depth`. Relations of $K$ in higher degrees are missed, so the result can only be too large; generators of $K$ above `depth` are missed as well. For Lee–Yang every $\operatorname{Ext}^1$ vanishes (rows: $M$, columns: $N$):

In [19]:
[[M.ext1_up_to(N, 4, pi).dimension() for N in simples] for M, pi in zip(simples, covers)]

[[0, 0], [0, 0]]

The Loewy layers of a module are the layers of its radical series ($\operatorname{rad} X$ is the intersection of the kernels of all maps from $X$ to simple modules; head first) and of its socle series (socle first). `radical_series_up_to(depth, simples)` and `socle_series_up_to(depth, simples)` give each layer as the multiplicities of the modules in `simples`; maps are found up to `depth`, so check stability in `depth`. A direct sum of simple modules is its own head and its own socle: one layer.

In [20]:
S = simples[0].direct_sum(simples[1])
S.radical_series_up_to(4, simples), S.socle_series_up_to(4, simples)

([[1, 1]], [[1, 1]])

## 6. A non-split extension: a Jordan block of $L_0$

The smallest non-split example in the library lives over the universal Virasoro algebra at $c = \frac37$. The Verma module $M(2)$ is simple there: the Kac determinant has no zero at $c = \frac37$, $h = 2$, and the code finds no maximal submodule up to depth 6. Its Zhu algebra is $\mathbb C[x]$, $x = [L]$; take the quotient $A_0 = \mathbb C[x]/((x-2)^2)$ and its module $U = A_0$, on which $[L]$ acts by a Jordan block. $U$ is projective over $A_0$, so the module $P = \bar M(U)$ induced from it (`V.induced_module(U)`) is projective in the category of positive energy modules with weights in $2 + \mathbb Z_{\ge 0}$ whose weight-2 space is an $A_0$-module, that is, $(L_0 - 2)^2 = 0$ there. $L_0$ has a Jordan block on its top:

In [21]:
V37 = va.Virasoro(QQ, 3/7, conformal_vector='L')
L37 = V37.gen('L')
A = V37.zhu_algebra(); x = A.gen(0)
A0 = A.quotient([(x - 2)^2])
P37 = V37.induced_module(A0.module({'L': matrix([[2, 1], [0, 2]])}))
M2 = V37.verma_module(2)
v = M2.highest_weight_vector()
[M2.maximal_submodule().weight_space(2 + d).dimension() for d in range(7)]

[0, 0, 0, 0, 0, 0, 0]

In [22]:
u0, u1 = P37.weight_space(2).basis()
L37.mode(u0, 0), L37.mode(u1, 0)

(2*u[0], u[0] + 2*u[1])

Module morphisms (`10_modules.ipynb`) show the structure of $P$. $\operatorname{Hom}(P, M(2))$ and $\operatorname{Hom}(M(2), P)$ are one-dimensional (the unpacking `[pi] = ...` checks it): $\pi$ sends $u_1 \mapsto v$ and kills $u_0$, and $j$ sends $v \mapsto u_0$. The kernel of $\pi$ and the image of $j$ have the same weight spaces: $P$ is an extension of $M(2)$ by $M(2)$, and $\operatorname{End}(P)$ is two-dimensional (the identity and $j \circ \pi$, the action of $L_0 - 2$ on the top).

In [23]:
[pi] = Hom(P37, M2).basis()
[j] = Hom(M2, P37).basis()
pi(u1), pi(u0), j(v)

(v, 0, u[0])

In [24]:
[pi.kernel().weight_space(2 + d).dimension() for d in range(5)], [j.image().weight_space(2 + d).dimension() for d in range(5)], len(End(P37).basis())

([1, 1, 2, 3, 5], [1, 1, 2, 3, 5], 2)

The Loewy layers of $P$ are $M(2) \mid M(2)$, and $\operatorname{Ext}^1(M(2), M(2))$, computed from the cover $\pi$, is one-dimensional in this category. `kernel_generators()` are the generators of $\ker\pi$, and `basis()` gives each class by the images of these generators. The dimension is the same for `depth` 2, 3, 4:

In [25]:
P37.radical_series_up_to(3, [M2]), P37.socle_series_up_to(3, [M2])

([[1], [1]], [[1], [1]])

In [26]:
E = M2.ext1_up_to(M2, 3, pi)
E.kernel_generators(), E.basis(), [M2.ext1_up_to(M2, d, pi).dimension() for d in (2, 3, 4)]

([u[0]], [(v,)], [1, 1, 1])

`E.extension(c, maps=True)` builds the extension module of a class $c$ (the pushout of the cover along the map $\ker\pi \to N$ of the class) with the maps $N \to E \to M$. For the basis class it has the weight spaces of $P$, a Jordan block of $L_0$ on its top (so it does not split), and the same Loewy layers. As $\operatorname{Ext}^1$ is one-dimensional, it is isomorphic to $P$. Its vectors print as `[0] ...` and `[1] ...`, the components in the direct sum $M(2) \oplus P$ it is a quotient of.

In [27]:
X, iota, proj = E.extension([1], maps=True)
a, b = X.weight_space(2).basis()
[X.weight_space(2 + d).dimension() for d in range(4)], [P37.weight_space(2 + d).dimension() for d in range(4)], (L37.mode(a, 0), L37.mode(b, 0)), X.socle_series_up_to(3, [M2])

([2, 2, 4, 6], [2, 2, 4, 6], (2*[0] v, [0] v + 2*[1] u[1]), [[1], [1]])

In [28]:
iota(v), proj(b)

([0] v, v)

The logarithmic example of this section, with simple modules, is the triplet algebra $\mathcal W(2)$: its projective covers $\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$ and $\mathcal P(1)$ (through the level-1 Zhu algebra), their Loewy layers, $\operatorname{Ext}^1$ between its simple modules, and fusion products of the projective modules, are in `15_triplet_modules.ipynb`, section 10.

## 7. Not there yet

* $M \boxtimes N$ itself as a module: only its depth-$d$ quotients are computed; `zhu_algebra_module` and `induced_module` give a module with the same top when that is enough.
* Fusion products over lattice vertex algebras (their answer $V_{\lambda+L} \boxtimes V_{\mu+L} = V_{\lambda+\mu+L}$ is known, but the algorithm needs a finite set of strong generators), modules with infinite-dimensional weight spaces, twisted sectors.
* A test that a window is large enough: the stability checks in `max_depth`, `max_weight` and `depth` are the user's.
* $\operatorname{Ext}^n$ for $n \ge 2$; $\operatorname{Ext}^1$ needs a projective cover (from `projective_cover` over a quotient with finite-dimensional Zhu algebras, or one the user claims is projective).

## References

- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X) — the Lee–Yang and Ising models (sections 1, 3).
- [BVY2019] K. Barron, N. Vander Werf, J. Yang, *Higher level Zhu algebras and modules for vertex operator algebras*, J. Pure Appl. Algebra 223 (2019), 3295–3317, [arXiv:1710.04767](https://arxiv.org/abs/1710.04767) — higher level Zhu algebras (section 5).
- [DLM1998] C. Dong, H. Li, G. Mason, *Vertex operator algebras and associative algebras*, J. Algebra 206 (1998), 67–96, [arXiv:q-alg/9612010](https://arxiv.org/abs/q-alg/9612010) — the universal modules $\bar M_n(U)$ and the adjunction (section 5).
- [GK1996] M. R. Gaberdiel, H. G. Kausch, *Indecomposable fusion products*, Nuclear Phys. B 477 (1996), 293–318, [arXiv:hep-th/9604026](https://arxiv.org/abs/hep-th/9604026) — the Nahm–Gaberdiel–Kausch algorithm and the fusion product at $c = -2$ (sections 1, 2, 4).
- [He2017] X. He, *Higher level Zhu algebras are subquotients of universal enveloping algebras*, J. Algebra 491 (2017), 265–279, [arXiv:1712.05671](https://arxiv.org/abs/1712.05671) — the model of the level-$n$ Zhu algebras used (section 5).
- [KR2019] S. Kanade, D. Ridout, *NGK and HLZ: fusion for physicists and mathematicians*, in: Affine, vertex and W-algebras, Springer INdAM Ser. 37 (2019), 135–181, [arXiv:1812.10713](https://arxiv.org/abs/1812.10713) — the conventions of the fusion computation (sections 1, 4).
- [Nah1994] W. Nahm, *Quasi-rational fusion products*, Internat. J. Modern Phys. B 8 (1994), 3693–3702, [arXiv:hep-th/9402039](https://arxiv.org/abs/hep-th/9402039) — the depth quotients and the special subspace (sections 1, 2).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212) — the minimal models are rational (section 5).
- [Wei1994] C. A. Weibel, *An introduction to homological algebra*, Cambridge Stud. Adv. Math. 38, Cambridge University Press (1994), [doi:10.1017/CBO9781139644136](https://doi.org/10.1017/CBO9781139644136) — $\operatorname{Ext}^1$ from a projective cover (section 5).
- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8) — the Zhu algebra (section 1).